# Introduction to Text Mining

How has the way we analyze text changed over time?

We look at visitor reviews of the Colosseum in Rome to compare two approaches.

| Era | Approach | Concept |
|-----|----------|----------|
| Classic | TF-IDF | Words as weighted counts |
| Modern | Embeddings | Text as a point in meaning-space |

### Download Colosseum reviews

In [1]:
# # Download text dataset and install requirements
# !curl -L -o ./rome-colosseum-visitor-reviews.zip https://www.kaggle.com/api/v1/datasets/download/uradkr/rome-colosseum-visitor-reviews
# !unzip rome-colosseum-visitor-reviews.zip
# !rm rome-colosseum-visitor-reviews.zip


### Install requirements

In [2]:
# !pip install sentence-transformers umap-learn plotly pandas

### Import libraries

In [3]:
from sentence_transformers import SentenceTransformer
import plotly.express as px
import umap
import pandas as pd
import re

/home/jelicic/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/jelicic/.local/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (1.26.20) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


### Helper functions


In [ ]:
def clean_text(text):
    regexp = r"[^\w\s]"
    text = re.sub(regexp,'',text).strip().lower()
    return text

def sort_dict_by_values(d):
    return {k: v for k, v in sorted(d.items(), key=lambda item: item[1],reverse=True)}
    

In [25]:
df = pd.read_csv('rome_colosseum_visitor_reviews_final.csv')

print(df.shape[0])


df

8285


,published_date,travel_month,published_platform,title,text,rating,tripType,helpful_votes,word_count,review_length_tier,sentiment_label,published_year,published_month,travel_season,is_verified_travel
0,2019-05-11,2019-05,OTHER,Colosseum,"A must see for any visitor to Rome, but go to ...",5,COUPLES,2,90,Long (75-200w),Positive,2019,5,Spring,True
1,2019-05-11,2019-04,OTHER,Amazing Structure and Architecture from The Past,"It is amazing structure built in Roman empire,...",5,COUPLES,2,507,Very Long (200w+),Positive,2019,5,Spring,True
2,2019-05-11,2019-05,MOBILE,C,Wonderful experience. We were blessed with a p...,5,COUPLES,2,20,Short (<25w),Positive,2019,5,Spring,True
3,2019-05-11,2019-04,MOBILE,SPECTACULAR,"Ok the Colosseum is an amazing place, iconic a...",5,FAMILY,2,149,Long (75-200w),Positive,2019,5,Spring,True
4,2019-05-11,2018-06,OTHER,More Amazing in Real Life!,The first sight of the Colosseum is overwhelmi...,5,FAMILY,0,59,Medium (25-75w),Positive,2019,5,Summer,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8280,2026-06-17,2026-05,OTHER,Book Direct for the Arena Ticket,We visited the Colosseum using the Full Experi...,5,COUPLES,0,194,Long (75-200w),Positive,2026,6,Spring,True
8281,2026-06-19,2026-06,MOBILE,Family Tour,The site is a must-do; even if the crowds are ...,5,FAMILY,0,59,Medium (25-75w),Positive,2026,6,Summer,True
8282,2026-06-19,2026-06,OTHER,Step back in time,Amazing to experience this walk round would pr...,5,COUPLES,0,17,Short (<25w),Positive,2026,6,Summer,True
8283,2026-06-20,2026-06,OTHER,A must visit,Fantastic experience full of history with ever...,5,COUPLES,0,36,Medium (25-75w),Positive,2026,6,Summer,True


In [24]:
texts = df['text'].apply(clean_text)

print(texts.values[-1])

brilliant experience booked through get your guide which was very easy and our guide aphrodite was an excellent guide that pointed out all the main feat of the areas we visited would use get your guide again without doubt


## Finding Important Words

If a word appears often in a document, it is probably important to that document.
But words like *the*, *and*, *is* appear in every document and carry no signal.

**TF-IDF** penalizes words that are common across all documents:

$$\text{TF-IDF}(word, doc) = \underbrace{\text{count in doc}}_{\text{Term Frequency}} \times \underbrace{\frac{1}{\text{count across all docs}}}_{\text{Inverse Document Frequency}}$$

A word scores high only when it is frequent in this document and rare everywhere else.

### Step 1: Document Frequency

How many reviews contain each word? Words that appear in many reviews carry less signal.

In [ ]:


doc_freq = dict()

for text in texts:
    for word in text.split(' '):
        if doc_freq.get(word):
            doc_freq[word]+=1 
        else:
            doc_freq[word] = 1
        
doc_freq

#facn sorting
sort_dict_by_values(doc_freq)


{'the': 34340,
 'to': 17429,
 'and': 17256,
 'a': 13804,
 'of': 10915,
 'in': 8809,
 'it': 8426,
 'you': 7999,
 'is': 7730,
 'was': 7632,
 'we': 7587,
 '': 7507,
 'tour': 5650,
 'for': 5123,
 'i': 5007,
 'colosseum': 4957,
 'this': 3850,
 'with': 3825,
 'but': 3647,
 'that': 3632,
 'as': 3246,
 'so': 3110,
 'on': 3103,
 'there': 2941,
 'rome': 2899,
 'not': 2868,
 'are': 2867,
 'guide': 2825,
 'at': 2749,
 'time': 2619,
 'be': 2612,
 'visit': 2604,
 'see': 2549,
 'get': 2539,
 'very': 2531,
 'were': 2512,
 'have': 2454,
 'tickets': 2430,
 'our': 2387,
 'history': 2334,
 'its': 2291,
 'an': 2279,
 'if': 2046,
 'from': 1957,
 'place': 1928,
 'had': 1926,
 'all': 1878,
 'go': 1852,
 'can': 1846,
 'your': 1791,
 'ticket': 1759,
 'they': 1689,
 'line': 1636,
 'amazing': 1604,
 'one': 1560,
 'which': 1539,
 'would': 1533,
 'must': 1473,
 'just': 1465,
 'worth': 1464,
 'when': 1434,
 'people': 1414,
 'forum': 1386,
 'my': 1379,
 'about': 1368,
 'inside': 1361,
 'experience': 1357,
 'around': 

### Step 2: Term Frequency

For each review, count how often each word appears.

In [7]:
term_freq = []

for doc in texts:
    tokenized = dict()
    for word in doc.split(' '):  # fixed: was 'text' (outer loop variable leak)
        if tokenized.get(word):
            tokenized[word] += 1
        else:
            tokenized[word] = 1
    term_freq.append(tokenized)

sort_dict_by_values(term_freq[-1])

{'guide': 4,
 'get': 2,
 'your': 2,
 'was': 2,
 'the': 2,
 'brilliant': 1,
 'experience': 1,
 'booked': 1,
 'through': 1,
 'which': 1,
 'very': 1,
 'easy': 1,
 'and': 1,
 'our': 1,
 'aphrodite': 1,
 'an': 1,
 'excellent': 1,
 'that': 1,
 'pointed': 1,
 'out': 1,
 'all': 1,
 'main': 1,
 'feat': 1,
 'of': 1,
 'areas': 1,
 'we': 1,
 'visited': 1,
 'would': 1,
 'use': 1,
 'again': 1,
 'without': 1,
 'doubt': 1}

### Step 3: TF-IDF Score

Multiply term frequency by inverse document frequency. Words that appear in many reviews score low.

In [8]:
tf_idf = []

for tf in term_freq:
    doc_tf_idf = dict()
    for word in tf:
        doc_tf_idf[word] =  round(tf[word] * (1/doc_freq[word]),4)
    
    tf_idf.append(doc_tf_idf)

sort_dict_by_values(tf_idf[-1])

{'aphrodite': 0.3333,
 'feat': 0.0526,
 'pointed': 0.0357,
 'doubt': 0.0294,
 'brilliant': 0.0101,
 'main': 0.0075,
 'areas': 0.004,
 'use': 0.004,
 'excellent': 0.0035,
 'easy': 0.0034,
 'again': 0.003,
 'without': 0.0019,
 'visited': 0.0017,
 'booked': 0.0014,
 'guide': 0.0014,
 'through': 0.0012,
 'your': 0.0011,
 'out': 0.0011,
 'get': 0.0008,
 'experience': 0.0007,
 'would': 0.0007,
 'which': 0.0006,
 'all': 0.0005,
 'very': 0.0004,
 'our': 0.0004,
 'an': 0.0004,
 'was': 0.0003,
 'that': 0.0003,
 'and': 0.0001,
 'the': 0.0001,
 'of': 0.0001,
 'we': 0.0001}

### What TF-IDF tells us

`aphrodite` scores **0.33** because it appears multiple times in this review but almost nowhere else across 8,000+ reviews. It is the word that sets this document apart.

`the`, `and`, `of` score near **0** because they appear in every review.

**Limitation**: TF-IDF treats text as a bag of words. It does not know that *"amazing"* and *"incredible"* are synonyms, or that *"not worth it"* is negative.

## Words that Define Positive and Negative Reviews

TF-IDF scores tell us which words matter to a single document.
If we add those scores across a group of reviews, we get the words most associated with that group.

In [9]:
ratings = df['rating'].values
pos_idx = [i for i, r in enumerate(ratings) if r >= 4]
neg_idx = [i for i, r in enumerate(ratings) if r <= 2]

def top_group_words(indices, tf_idf_list, min_doc_freq=5):
    agg = {}
    for i in indices:
        for word, score in tf_idf_list[i].items():
            if word and doc_freq.get(word, 0) >= min_doc_freq:
                agg[word] = agg.get(word, 0) + score
    return sort_dict_by_values(agg)

pos_top = top_group_words(pos_idx, tf_idf)
neg_top = top_group_words(neg_idx, tf_idf)

num_display = 10

print(f"Positive reviews ({len(pos_idx)}, rated 4-5 stars)")
for word, score in list(pos_top.items())[:num_display]:
    print(f"\t{word:20s} {score:.4f}")

print(f"\nNegative reviews ({len(neg_idx)}, rated 1-2 stars)")
for word, score in list(neg_top.items())[:num_display]:
    print(f"\t{word:20s} {score:.4f}")

Positive reviews (7472, rated 4-5 stars)
	masterpiece          1.0013
	titus                1.0011
	disappoint           1.0011
	tips                 1.0010
	monumental           1.0010
	constructed          1.0010
	spectators           1.0010
	powerful             1.0009
	society              1.0009
	pleased              1.0009

Negative reviews (360, rated 1-2 stars)
	245                  1.0000
	unacceptable         0.8888
	corridor             0.8571
	girlfriend           0.8000
	racist               0.8000
	parties              0.8000
	nonexistent          0.8000
	message              0.7500
	cancel               0.7145
	upset                0.7145


## Modern Approach: Text Embeddings

TF-IDF treats text as a bag of words. It does not know that **"amazing"** and **"incredible"** mean the same thing, or that **"not bad"** is positive.

Language models turn each piece of text into a vector: a point in high-dimensional space where similar meanings end up close together.

We use `all-MiniLM-L6-v2` to embed the reviews, then reduce to 2D to see the structure.

In [10]:
model = SentenceTransformer('all-MiniLM-L6-v2')

/home/jelicic/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13930.97it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
# Each review -> a 384-dimensional vector 
texts = df['text'].fillna('').tolist()
embeddings = model.encode(texts, show_progress_bar=True, batch_size=64)


Batches: 100%|██████████| 130/130 [00:48<00:00,  2.70it/s]


In [18]:
# Reduce 384 dimensions -> 2 dimensions with UMAP
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=10)
embeddings_2d = reducer.fit_transform(embeddings)

print(f"Reduced: {embeddings.shape} -> {embeddings_2d.shape}")

/home/jelicic/.local/lib/python3.10/site-packages/umap/umap_.py:1943: UserWarning:

n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.



Reduced: (8285, 384) -> (8285, 2)


In [19]:
import textwrap

def wrap_text(text, width=55):
    return '<br>'.join(textwrap.wrap(str(text), width=width))

plot_df = pd.DataFrame({
    'x': embeddings_2d[:, 0],
    'y': embeddings_2d[:, 1],
    'rating': df['rating'].values,
    'review': df['text'].fillna('').str[:400].apply(wrap_text).values,
})

fig = px.scatter(
    plot_df,
    x='x', y='y',
    color='rating',
    color_continuous_scale='RdYlGn',
    range_color=[1, 5],
    custom_data=['rating', 'review'],
    title='Colosseum Reviews',
    opacity=0.7,
)

fig.update_traces(
    marker=dict(size=6),
    hovertemplate='<b>%{customdata[0]} star(s)</b><br><br>%{customdata[1]}<extra></extra>',
)

fig.update_layout(
    xaxis=dict(showticklabels=False, title=''),
    yaxis=dict(showticklabels=False, title=''),
    margin=dict(l=40, r=160, t=50, b=40),
    coloraxis_colorbar=dict(
        tickvals=[1, 2, 3, 4, 5],
        ticktext=['1 star', '2 stars', '3 stars', '4 stars', '5 stars'],
        title='Rating',
    ),
)

fig.show()